# BigQuery Anti-Pattern Recognition - UDF Demo

This notebook demonstrates how to use the BigQuery Anti-Pattern Recognition tool as a BigQuery User-Defined Function (UDF).

## What this notebook does:
1. Load configuration from the setup notebook
2. Test the BigQuery remote function
3. Analyze queries directly in SQL
4. Process INFORMATION_SCHEMA data
5. Create monitoring views and dashboards
6. Demonstrate advanced SQL integration

## Prerequisites:
- Complete `01_setup_and_deploy.ipynb` first
- BigQuery remote function must be created

---

## Step 1: Load Configuration and Setup

In [ ]:
# Import required libraries
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from google.cloud import bigquery

# Import our utilities
from utils import (
    ConfigManager, 
    AntiPatternAnalyzer, 
    ResultsFormatter, 
    SampleQueries,
    format_sql
)

print("✅ Libraries imported successfully!")

In [ ]:
# Load configuration from setup notebook
config = ConfigManager()

# Verify deployment is complete
if config.get('deployment_status') != 'completed':
    print("❌ Setup not complete. Please run 01_setup_and_deploy.ipynb first.")
else:
    print("✅ Configuration loaded successfully!")
    print(f"UDF Name: {config.get('udf_name')}")
    print(f"Project: {config.get('project_id')}")
    print(f"Dataset: {config.get('bq_dataset')}")

# Initialize BigQuery client
try:
    bq_client = bigquery.Client(project=config.get('project_id'))
    print("✅ BigQuery client initialized!")
except Exception as e:
    print(f"❌ Failed to initialize BigQuery client: {e}")
    bq_client = None

## Step 2: Test the Remote Function

Let's test our BigQuery remote function with simple queries:

In [ ]:
# Test the UDF with a simple query
if bq_client:
    print("🧪 Testing BigQuery Remote Function")
    print("=" * 50)
    
    # Simple test query
    test_sql = f"""
    SELECT 
        'SELECT * FROM dataset.table ORDER BY 1' as test_query,
        {config.get('udf_name')}('SELECT * FROM dataset.table ORDER BY 1') as antipatterns
    """
    
    try:
        print("Executing test query...")
        query_job = bq_client.query(test_sql)
        results = query_job.result()
        
        for row in results:
            print(f"\n📝 Test Query: {row.test_query}")
            print(f"🔍 Anti-patterns: {row.antipatterns}")
            
            # Parse the JSON response
            if row.antipatterns:
                try:
                    ap_data = json.loads(row.antipatterns)
                    if 'antipatterns' in ap_data:
                        print(f"\n✅ Found {len(ap_data['antipatterns'])} anti-patterns:")
                        for i, ap in enumerate(ap_data['antipatterns'], 1):
                            print(f"  {i}. {ap.get('name', 'Unknown')}: {ap.get('result', 'No description')}")
                    else:
                        print("✅ No anti-patterns detected!")
                except json.JSONDecodeError:
                    print(f"⚠️ Could not parse response: {row.antipatterns}")
        
        print("\n✅ Remote function test completed successfully!")
        
    except Exception as e:
        print(f"❌ Error testing remote function: {e}")
        print("\nTroubleshooting tips:")
        print("1. Verify the remote function was created successfully")
        print("2. Check that the Cloud Run service is running")
        print("3. Ensure proper permissions are granted")
else:
    print("❌ BigQuery client not available")

## Step 3: Interactive SQL Query Tester

Create an interface to test different queries using the UDF:

In [ ]:
# Interactive UDF tester
sample_queries = SampleQueries.get_all_queries()

# Widgets for query selection and input
query_selector = widgets.Dropdown(
    options=[('Custom Query', 'custom')] + [(v['name'], k) for k, v in sample_queries.items()],
    value='select_star',
    description='Select Query:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

query_input = widgets.Textarea(
    value=sample_queries['select_star']['query'],
    description='SQL Query:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='100%', height='150px')
)

test_udf_button = widgets.Button(
    description='🔍 Test with UDF',
    button_style='primary',
    layout=widgets.Layout(width='200px')
)

udf_output_area = widgets.Output()

def on_query_change(change):
    if change['new'] != 'custom':
        query_input.value = sample_queries[change['new']]['query']

def on_test_udf_click(b):
    with udf_output_area:
        clear_output()
        print("🔄 Testing query with BigQuery UDF...")
        
        if not bq_client:
            print("❌ BigQuery client not available")
            return
        
        try:
            # Escape single quotes in the query
            escaped_query = query_input.value.replace("'", "\\'").replace('"', '\\"')
            
            # Create SQL to call the UDF
            udf_sql = f"""
            SELECT 
                '{escaped_query}' as original_query,
                {config.get('udf_name')}('{escaped_query}') as antipatterns
            """
            
            print("Executing UDF query...")
            query_job = bq_client.query(udf_sql)
            results = query_job.result()
            
            for row in results:
                print("\n📊 UDF Analysis Results:")
                print("=" * 50)
                
                if row.antipatterns:
                    try:
                        ap_data = json.loads(row.antipatterns)
                        if 'antipatterns' in ap_data and ap_data['antipatterns']:
                            print(f"\n🔍 Found {len(ap_data['antipatterns'])} anti-patterns:")
                            for i, ap in enumerate(ap_data['antipatterns'], 1):
                                print(f"\n{i}. {ap.get('name', 'Unknown')}")
                                print(f"   {ap.get('result', 'No description')}")
                        else:
                            print("✅ No anti-patterns detected!")
                    except json.JSONDecodeError as e:
                        print(f"⚠️ Could not parse UDF response: {e}")
                        print(f"Raw response: {row.antipatterns}")
                else:
                    print("⚠️ No response from UDF")
            
            print("\n✅ UDF test completed!")
            
        except Exception as e:
            print(f"❌ Error testing UDF: {e}")
            print("\nSQL that was attempted:")
            print(udf_sql)

query_selector.observe(on_query_change, names='value')
test_udf_button.on_click(on_test_udf_click)

# Display interface
display(widgets.VBox([
    widgets.HTML("<h3>🔍 BigQuery UDF Tester</h3>"),
    query_selector,
    query_input,
    test_udf_button,
    udf_output_area
]))

## Step 4: Batch Analysis with UDF

Let's analyze multiple queries at once using the UDF:

In [ ]:
# Batch analysis using UDF
if bq_client:
    print("🔄 Batch Analysis with BigQuery UDF")
    print("=" * 50)
    
    # Create a table with sample queries for batch processing
    sample_queries = SampleQueries.get_all_queries()
    
    # Build SQL for batch analysis
    query_cases = []
    for key, query_info in sample_queries.items():
        escaped_query = query_info['query'].replace("'", "\\'").replace('"', '\\"')
        query_cases.append(f"('{key}', '{query_info['name']}', '{escaped_query}')")
    
    batch_sql = f"""
    WITH sample_queries AS (
        SELECT * FROM UNNEST([
            {', '.join(query_cases)}
        ]) AS t(query_key, query_name, query_text)
    )
    SELECT 
        query_key,
        query_name,
        query_text,
        {config.get('udf_name')}(query_text) as antipatterns
    FROM sample_queries
    ORDER BY query_key
    """
    
    try:
        print("Executing batch analysis...")
        query_job = bq_client.query(batch_sql)
        results = query_job.result()
        
        batch_results = []
        
        for row in results:
            antipattern_count = 0
            antipattern_names = []
            
            if row.antipatterns:
                try:
                    ap_data = json.loads(row.antipatterns)
                    if 'antipatterns' in ap_data and ap_data['antipatterns']:
                        antipattern_count = len(ap_data['antipatterns'])
                        antipattern_names = [ap.get('name', 'Unknown') for ap in ap_data['antipatterns']]
                except json.JSONDecodeError:
                    pass
            
            batch_results.append({
                'query_key': row.query_key,
                'query_name': row.query_name,
                'antipattern_count': antipattern_count,
                'antipattern_names': antipattern_names
            })
            
            print(f"✅ {row.query_name}: {antipattern_count} anti-patterns")
        
        # Create DataFrame for visualization
        df_batch = pd.DataFrame(batch_results)
        
        print(f"\n📊 Batch Analysis Summary:")
        print(f"Total queries analyzed: {len(df_batch)}")
        print(f"Total anti-patterns found: {df_batch['antipattern_count'].sum()}")
        print(f"Average anti-patterns per query: {df_batch['antipattern_count'].mean():.1f}")
        
        # Display results table
        display(HTML("<h3>📋 Batch Analysis Results</h3>"))
        display(df_batch[['query_name', 'antipattern_count', 'antipattern_names']])
        
        # Create visualization
        if not df_batch.empty:
            fig = px.bar(
                df_batch,
                x='query_name',
                y='antipattern_count',
                title='Anti-Patterns Found by Query (UDF Analysis)',
                labels={'antipattern_count': 'Number of Anti-Patterns', 'query_name': 'Query'},
                color='antipattern_count',
                color_continuous_scale='Reds'
            )
            fig.update_xaxis(tickangle=45)
            fig.show()
        
    except Exception as e:
        print(f"❌ Error in batch analysis: {e}")
        print("\nSQL that was attempted:")
        print(batch_sql[:500] + "..." if len(batch_sql) > 500 else batch_sql)
else:
    print("❌ BigQuery client not available")

## Step 5: SQL Examples for Direct Use

Here are some SQL examples you can run directly in BigQuery console:

In [ ]:
# Generate SQL examples for direct use
print("📝 SQL Examples for BigQuery Console")
print("=" * 50)

udf_name = config.get('udf_name')

sql_examples = {
    "Simple Query Analysis": f"""
-- Analyze a single query
SELECT 
    'SELECT * FROM dataset.table ORDER BY 1' as query,
    {udf_name}('SELECT * FROM dataset.table ORDER BY 1') as antipatterns;
""",
    
    "Batch Query Analysis": f"""
-- Analyze multiple queries at once
WITH queries_to_analyze AS (
    SELECT * FROM UNNEST([
        ('query1', 'SELECT * FROM table1'),
        ('query2', 'SELECT col1 FROM table2 ORDER BY col1'),
        ('query3', 'SELECT * FROM table3 WHERE col LIKE \"%pattern%\"')
    ]) AS t(query_id, query_text)
)
SELECT 
    query_id,
    query_text,
    {udf_name}(query_text) as antipatterns
FROM queries_to_analyze;
""",
    
    "INFORMATION_SCHEMA Analysis": f"""
-- Analyze recent expensive queries
WITH recent_expensive_queries AS (
    SELECT 
        job_id,
        user_email,
        query,
        total_slot_ms,
        total_bytes_processed
    FROM `{config.get('project_id')}.region-{config.get('region')}.INFORMATION_SCHEMA.JOBS`
    WHERE DATE(creation_time) >= DATE_SUB(CURRENT_DATE(), INTERVAL 7 DAYS)
        AND statement_type = 'SELECT'
        AND state = 'DONE'
        AND total_slot_ms > 10000  -- Focus on expensive queries
    ORDER BY total_slot_ms DESC
    LIMIT 10
)
SELECT 
    job_id,
    user_email,
    LEFT(query, 100) as query_preview,
    total_slot_ms,
    {udf_name}(query) as antipatterns
FROM recent_expensive_queries;
""",
    
    "Anti-Pattern Summary": f"""
-- Create a summary of anti-patterns found
WITH analyzed_queries AS (
    SELECT 
        job_id,
        query,
        {udf_name}(query) as antipatterns_json
    FROM `{config.get('project_id')}.region-{config.get('region')}.INFORMATION_SCHEMA.JOBS`
    WHERE DATE(creation_time) = CURRENT_DATE()
        AND statement_type = 'SELECT'
        AND state = 'DONE'
    LIMIT 50  -- Limit for demo
),
antipatterns_extracted AS (
    SELECT 
        job_id,
        JSON_EXTRACT_ARRAY(antipatterns_json, '$.antipatterns') as antipatterns_array
    FROM analyzed_queries
    WHERE antipatterns_json IS NOT NULL
),
flattened_antipatterns AS (
    SELECT 
        job_id,
        JSON_EXTRACT_SCALAR(antipattern, '$.name') as antipattern_name
    FROM antipatterns_extracted,
    UNNEST(antipatterns_array) as antipattern
)
SELECT 
    antipattern_name,
    COUNT(*) as occurrence_count,
    COUNT(DISTINCT job_id) as affected_queries
FROM flattened_antipatterns
GROUP BY antipattern_name
ORDER BY occurrence_count DESC;
"""
}

for title, sql in sql_examples.items():
    print(f"\n### {title}")
    print("```sql")
    print(sql.strip())
    print("```")

print("\n💡 Copy and paste these examples into BigQuery console to run them!")

## Step 6: Export Results and Summary

Create a summary of the UDF demo:

In [ ]:
# Create UDF demo summary
summary_html = f"""
<div style="background-color: #f0f8ff; padding: 20px; border-radius: 10px; margin: 10px 0;">
    <h3>📋 BigQuery UDF Demo Summary</h3>
    
    <h4>🔧 UDF Configuration:</h4>
    <ul>
        <li><strong>Function Name:</strong> {config.get('udf_name')}</li>
        <li><strong>Project:</strong> {config.get('project_id')}</li>
        <li><strong>Dataset:</strong> {config.get('bq_dataset')}</li>
        <li><strong>Cloud Run Service:</strong> {config.get('service_url')}</li>
    </ul>
    
    <h4>✅ What You Can Do Now:</h4>
    <ul>
        <li><strong>Direct SQL Usage:</strong> Call the UDF directly in BigQuery console</li>
        <li><strong>Batch Analysis:</strong> Analyze multiple queries in a single SQL statement</li>
        <li><strong>INFORMATION_SCHEMA Integration:</strong> Analyze historical queries automatically</li>
        <li><strong>Monitoring Views:</strong> Create views for ongoing anti-pattern monitoring</li>
        <li><strong>Scheduled Queries:</strong> Set up automated analysis of new queries</li>
    </ul>
    
    <h4>🔍 Example Usage:</h4>
    <pre style="background-color: #f5f5f5; padding: 10px; border-radius: 5px;">
-- Simple usage
SELECT {config.get('udf_name')}('SELECT * FROM dataset.table') as antipatterns;

-- With INFORMATION_SCHEMA
SELECT 
    job_id,
    query,
    {config.get('udf_name')}(query) as antipatterns
FROM `{config.get('project_id')}.region-{config.get('region')}.INFORMATION_SCHEMA.JOBS`
WHERE DATE(creation_time) = CURRENT_DATE()
    AND statement_type = 'SELECT'
LIMIT 10;
    </pre>
    
    <h4>🚀 Next Steps:</h4>
    <ul>
        <li>Run <strong>04_streamlit_frontend.ipynb</strong> to deploy the web interface</li>
        <li>Set up scheduled queries for continuous monitoring</li>
        <li>Create dashboards in Looker Studio or Data Studio</li>
        <li>Integrate with your existing data pipelines</li>
    </ul>
    
    <h4>💡 Pro Tips:</h4>
    <ul>
        <li>Use the UDF in views to create reusable analysis patterns</li>
        <li>Combine with BigQuery's ML functions for predictive analysis</li>
        <li>Set up alerts based on anti-pattern detection results</li>
        <li>Use in stored procedures for automated optimization workflows</li>
    </ul>
</div>
"""

display(HTML(summary_html))

print("\n🎯 BigQuery UDF demo completed successfully!")
print("\nThe UDF is now ready for production use. You can:")
print("1. Use it directly in any BigQuery SQL query")
print("2. Integrate it into your data pipelines")
print("3. Create monitoring dashboards")
print("4. Set up automated analysis workflows")
print("5. Proceed to the Streamlit frontend demo")